#  Word Predictor (Advanced LSTM)


In [1]:
import sys
!{sys.executable} -m pip install torch tokenizers

  Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached setuptools-83.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   - -------------------------------------- 3.1/122.1 MB 15.8 MB/s eta 0:00:08
   - -------------------------------------- 6.0/122.1 MB 14.8 MB/s eta 0:00:08
   -- ------------------------------------- 8.1/122.1 MB 14.4 MB/s eta 0:00:08
   --- ------------------------------------ 11.0/122.1 MB 13.6 MB/s eta 0:00:09
   ---- -------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Why this works: sys.executable gets the exact path to the Python engine running your current notebook. By using it, we guarantee that pip installs torch right where the notebook can see it.

## Imports

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import numpy as np
import math
import os
import random

## Setup Device and Seeds

In [3]:
# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 1. Data Loading

In [4]:
data_path = 'task.txt'

if os.path.exists(data_path):
    with open(data_path, 'r', encoding='utf-8') as f:
        faqs = f.read()
else:
    # Fallback to the known text
    faqs = """About the Program
What is the course fee for Data Science Mentorship Program (DSMP 2023)
The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.
What is the total duration of the course?
The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)
What is the syllabus of the mentorship program?
We will be covering the following modules:
Python Fundamentals
Python libraries for Data Science
Data Analysis
SQL for Data Science
Maths for Machine Learning
ML Algorithms
Practical ML
MLOPs
Case studies
Will Deep Learning and NLP be a part of this program?
No, NLP and Deep Learning both are not a part of this program’s curriculum.
What if I miss a live session? Will I get a recording of the session?
Yes all our sessions are recorded, so even if you miss a session you can go back and watch the recording.
Where can I find the class schedule?
Roughly, all the sessions last 2 hours.
What is the language spoken by the instructor during the sessions?
Hinglish
How will I be informed about the upcoming class?
You will get a mail from our side before every paid session once you become a paid user.
Can I do this course if I am from a non-tech background?
Yes, absolutely.
I am late, can I join the program in the middle?
Absolutely, you can join the program anytime.
If I join/pay in the middle, will I be able to see all the past lectures?
Yes, once you make the payment you will be able to see all the past content in your dashboard.
Where do I have to submit the task?
You don’t have to submit the task. We will provide you with the solutions, you have to self evaluate the task yourself.
Will we do case studies in the program?
Yes.
Where can we contact you?
You can mail us at nitish.campusx@gmail.com
Payment/Registration related questions
Where do we have to make our payments? Your YouTube channel or website?
You have to make all your monthly payments on our website.
Can we pay the entire amount of Rs 5600 all at once?
Unfortunately no, the program follows a monthly subscription model.
What is the validity of monthly subscription?
The validity period is 30 days from the day you make the payment.
What if I don’t like the course after making the payment. What is the refund policy?
You get a 7 days refund period from the day you have made the payment.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmail.com
Post registration queries
Till when can I view the paid videos on the website?
You can watch the videos till your subscription is valid.
Why lifetime validity is not provided?
Because of the low course fee.
Where can I reach out in case of a doubt after the session?
You will have to fill a google form provided in your dashboard and our team will contact you for a 1 on 1 doubt clearance session
Certificate and Placement Assistance related queries
What is the criteria to get the certificate?
There are 2 criterias:
You have to pay the entire fee of Rs 5600
You have to attempt all the course assessments.
I am joining late. How can I pay payment of the earlier months?
You will get a link to pay fee of earlier months in your dashboard once you pay for the current month.
I have read that Placement assistance is a part of this program. What comes under Placement assistance?
This is to clarify that Placement assistance does not mean Placement guarantee.
Portfolio Building sessions
Soft skill sessions
Sessions with industry mentors
Discussion on Job hunting strategies"""

## BPE Tokenization

In [5]:
# Initialize a Byte-Pair Encoding tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

In [6]:
# Train the tokenizer
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"], vocab_size=1000)
tokenizer.train_from_iterator([faqs], trainer)

In [7]:
# Test tokenizer
encoded = tokenizer.encode("What is the course fee")
print(f"Vocab Size: {tokenizer.get_vocab_size()}")
print(f"Sample Encoding: {encoded.tokens}")

Vocab Size: 819
Sample Encoding: ['What', 'is', 'the', 'course', 'fee']


## 2. Dataset Preparation (Pre-Padding Invariant)

In [8]:
class FAQDataset(Dataset):
    def __init__(self, text, tokenizer, seq_length=15):
        self.tokenizer = tokenizer
        self.seq_length = seq_length
        self.pad_id = tokenizer.token_to_id("[PAD]")
        
        # Tokenize the entire corpus
        encoded = tokenizer.encode(text)
        self.tokens = encoded.ids
        
        self.sequences = []
        self.targets = []
        
        # Create sequences using sliding window
        for i in range(1, len(self.tokens)):
            # End index is i, target is i
            seq = self.tokens[max(0, i - self.seq_length) : i]
            target = self.tokens[i]
            
            # PRE-PADDING: pad with zeros at the BEGINNING
            if len(seq) < self.seq_length:
                num_pads = self.seq_length - len(seq)
                seq = [self.pad_id] * num_pads + seq
                
            self.sequences.append(seq)
            self.targets.append(target)
            
        self.sequences = torch.tensor(self.sequences, dtype=torch.long)
        self.targets = torch.tensor(self.targets, dtype=torch.long)
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

In [9]:
seq_length = 15
dataset = FAQDataset(faqs, tokenizer, seq_length=seq_length)
print(f"Created {len(dataset)} training sequences.")

Created 1076 training sequences.


In [10]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

## 3. Advanced AWD-LSTM Architecture

In [11]:
class LockedDropout(nn.Module):
    """Variational Dropout: applies the same dropout mask across the sequence length"""
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or not self.p:
            return x
        # x shape: (batch, seq_len, embed_size)
        m = x.data.new(x.size(0), 1, x.size(2)).bernoulli_(1 - self.p)
        mask = torch.autograd.Variable(m, requires_grad=False) / (1 - self.p)
        mask = mask.expand_as(x)
        return mask * x

In [12]:
class WeightDropLSTM(nn.Module):
    """Applies DropConnect to hidden-to-hidden weights of the LSTM"""
    def __init__(self, input_size, hidden_size, weight_dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.weight_dropout = weight_dropout
        
        # We need to manually apply DropConnect to the weight_hh matrices
        self.weights = ['weight_hh_l0']
        
        for w in self.weights:
            w_raw = getattr(self.lstm, w)
            delattr(self.lstm, w)
            self.register_parameter(w + '_raw', nn.Parameter(w_raw.data))

    def _setweights(self):
        for w in self.weights:
            raw_w = getattr(self, w + '_raw')
            # Apply dropout to the weight matrix
            if self.training and self.weight_dropout > 0:
                mask = F.dropout(torch.ones_like(raw_w), p=self.weight_dropout, training=True)
                w_dropped = raw_w * mask
            else:
                w_dropped = raw_w
            setattr(self.lstm, w, w_dropped)

    def forward(self, x, hx=None):
        self._setweights()
        return self.lstm(x, hx)

In [13]:
class MoSAWDLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_mixtures=3, dropout=0.3):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_size = embed_size
        self.hidden_size = hidden_size
        self.num_mixtures = num_mixtures
        
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lockdrop = LockedDropout(dropout)
        
        # AWD-LSTM
        self.rnn = WeightDropLSTM(embed_size, hidden_size, weight_dropout=dropout)
        
        # MoS Layer components
        self.prior = nn.Linear(hidden_size, num_mixtures, bias=False)
        self.latent = nn.Linear(hidden_size, num_mixtures * embed_size)
        self.decoder = nn.Linear(embed_size, vocab_size)
        
        # TIE WEIGHTS
        self.decoder.weight = self.embedding.weight
        
    def forward(self, x):
        emb = self.embedding(x)
        emb = self.lockdrop(emb)
        
        out, _ = self.rnn(emb)
        final_out = out[:, -1, :] 
        
        pi = F.softmax(self.prior(final_out), dim=-1)
        
        latent_h = self.latent(final_out).view(-1, self.num_mixtures, self.embed_size)
        latent_h = torch.tanh(latent_h)
        
        mixture_logits = self.decoder(latent_h)
        mixture_probs = F.softmax(mixture_logits, dim=-1)
        
        out_probs = torch.sum(pi.unsqueeze(-1) * mixture_probs, dim=1)
        return torch.log(out_probs + 1e-8)

## Model Instantiation

In [14]:
vocab_size = tokenizer.get_vocab_size()
embed_size = 256
hidden_size = 512

model = MoSAWDLSTM(vocab_size, embed_size, hidden_size, num_mixtures=3, dropout=0.3).to(device)
print(f"Total Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Total Parameters: 2182963


## 4. Training (OneCycleLR + Label Smoothing)

In [15]:
# Label Smoothing function
def smooth_loss(log_probs, target, epsilon=0.1):
    n_classes = log_probs.size(-1)
    # create smoothed targets
    one_hot = torch.zeros_like(log_probs).scatter(1, target.view(-1, 1), 1)
    one_hot = one_hot * (1 - epsilon) + (1 - one_hot) * epsilon / (n_classes - 1)
    # cross entropy with smoothed targets
    loss = -(one_hot * log_probs).sum(dim=1).mean()
    return loss

In [16]:
epochs = 35
criterion = nn.NLLLoss() # Since model outputs log_probs

In [17]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=5e-3, steps_per_epoch=len(dataloader), epochs=epochs
)

In [18]:
# Training Loop
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch_idx, (inputs, targets) in enumerate(dataloader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        log_probs = model(inputs)
        
        # Advanced: Label smoothed loss
        loss = smooth_loss(log_probs, targets, epsilon=0.1)
        
        loss.backward()
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.25)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(dataloader)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

Epoch 01/35 | Loss: 6.5288 | LR: 0.000307
Epoch 05/35 | Loss: 3.0226 | LR: 0.002431
Epoch 10/35 | Loss: 3.8153 | LR: 0.004976
Epoch 15/35 | Loss: 2.9486 | LR: 0.004590
Epoch 20/35 | Loss: 2.5933 | LR: 0.003355
Epoch 25/35 | Loss: 2.1470 | LR: 0.001780
Epoch 30/35 | Loss: 1.9267 | LR: 0.000491
Epoch 35/35 | Loss: 1.8667 | LR: 0.000000


## 5. Text Generation (with Continuous Cache Pointer)

In [19]:
def generate_text(prompt, max_len=50, temperature=0.7, cache_theta=0.15):
    model.eval()
    
    encoded = tokenizer.encode(prompt).ids
    words = encoded.copy()
    
    # Continuous Cache
    history_cache = encoded.copy()
    
    print(f"Prompt: {prompt}")
    print("Generated: ", end="")
    
    with torch.no_grad():
        for _ in range(max_len):
            # Pre-pad sequence
            seq = words[-seq_length:]
            if len(seq) < seq_length:
                seq = [tokenizer.token_to_id("[PAD]")] * (seq_length - len(seq)) + seq
                
            input_tensor = torch.tensor([seq], dtype=torch.long).to(device)
            
            # Get model log probs
            log_probs = model(input_tensor)
            probs = torch.exp(log_probs)[0] # (vocab_size)
            
            # Continuous Cache Pointer Mechanism
            if len(history_cache) > 0:
                cache_distribution = torch.zeros_like(probs)
                # Count frequencies in recent history
                for w_id in history_cache[-20:]: # look at last 20 words
                    if w_id < len(cache_distribution):
                        cache_distribution[w_id] += 1.0
                if cache_distribution.sum() > 0:
                    cache_distribution = cache_distribution / cache_distribution.sum()
                    
                # Interpolate Model Probs and Cache Probs
                probs = (1 - cache_theta) * probs + cache_theta * cache_distribution
            
            # Apply temperature scaling
            probs = torch.pow(probs, 1.0 / temperature)
            probs = probs / torch.sum(probs)
            
            # Sample from distribution
            next_word_id = torch.multinomial(probs, 1).item()
            
            words.append(next_word_id)
            history_cache.append(next_word_id)
            
            decoded_word = tokenizer.decode([next_word_id])
            print(decoded_word, end=" ")
            
            # Stop if we hit EOS or if we start looping terribly (basic check)
            if next_word_id == tokenizer.token_to_id("[EOS]"):
                break
                
    print()
    return tokenizer.decode(words)

## Generation Tests

In [20]:
print("\n--- Test 1 ---")
_ = generate_text("What is the total duration", max_len=15, temperature=0.5)


--- Test 1 ---
Prompt: What is the total duration
Generated: of the course ? The total duration of the course ? The total duration of 


In [21]:
print("\n--- Test 2 ---")
_ = generate_text("Will Deep Learning and", max_len=15, temperature=0.5)


--- Test 2 ---
Prompt: Will Deep Learning and
Generated: NLP be a part of this program ’ s curriculum . What if I miss 


In [22]:
print("\n--- Test 3 ---")
_ = generate_text("Where can I find the", max_len=15, temperature=0.5)


--- Test 3 ---
Prompt: Where can I find the
Generated: class schedule ? Checkout this google sheet to see month by month time table month 
